# How to use the shared code

Load the data with a split, score predictions with the shared `evaluate()`, and save them in the shared
format. No model is trained here: a constant baseline stands in for "your model".

**Setup (once):** `pip install -r requirements.txt` and `pip install -e .` (see the README). No other preparation:
the first `load_dataset()` call downloads the raw data, cleans it and checks it against the group's reference
(about 3 minutes). Later calls read the cached copy in seconds.

In [1]:
# pick up edits to src/ without restarting the kernel
%load_ext autoreload
%autoreload 2

## 1. Load the cleaned data with a split

`load_dataset(regime)` returns the cleaned articles plus a `split` column (`train` / `val` / `test`).
Model on `content`; the label is `is_fake` (1 = fake).

In [2]:
from fake_news.data import load_dataset

df = load_dataset("random")
print(df.shape)
df.head(3)

(62525, 7)


,doc_id,title,text,content,n_words,is_fake,split
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,law enforcement on high alert following threat...,889,1,val
1,1,,Did they post their votes for Hillary already?,did they post their votes for hillary already?,8,1,test
2,2,UNBELIEVABLE! OBAMA'S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",unbelievable! obama s attorney general says mo...,52,1,train


In [3]:
df.groupby("split").is_fake.agg(n="size", fake_rate="mean")

,n,fake_rate
split,,
test,9379,0.444077
train,43767,0.444125
val,9379,0.444184


Need less memory? Load only the columns you use (`doc_id`, `is_fake` and `split` always come along):

In [4]:
small = load_dataset("random", columns=["content"])
small.columns.tolist()

['doc_id', 'content', 'is_fake', 'split']

## 2. Your model goes here: fit on `train` only

A real model fits everything (vocabulary, scaler, classifier, ...) on the **training** rows only, for example:

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(train.content)   # fit on train only
X_val = vectorizer.transform(val.content)           # val is only transformed
clf = LogisticRegression(max_iter=1000, random_state=config.SEED).fit(X_train, train.is_fake)
prob_fake = clf.predict_proba(X_val)[:, 1]
```

To keep this notebook model-free, a constant baseline predicts the training fake rate for every article:

In [5]:
import numpy as np

train, val = df[df.split == "train"], df[df.split == "val"]
prob_fake = np.full(len(val), train.is_fake.mean())   # replace with your model's P(fake)

## 3. Score on `val` with the shared `evaluate()`

Same metrics for every model: accuracy, precision, recall, F1 (fake = positive class), ROC-AUC and the confusion
matrix. Tune on `val`; `test` is used once, at the very end, for all models together. (A constant prediction has
ROC-AUC 0.5 and predicts no article as fake at the 0.5 threshold.)

In [6]:
from fake_news.evaluation import evaluate

evaluate(val.is_fake, prob_fake)

{'n': 9379,
 'accuracy': 0.5558161850943597,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'tn': 5213,
 'fp': 0,
 'fn': 4166,
 'tp': 0}

## 4. Save predictions in the shared format

One file per model and split regime with columns `doc_id, split, prob_fake`, in
`data/processed/predictions/<model>__<regime>.csv`. Blending (Model D) joins these files on `doc_id`.

In [7]:
import pandas as pd

from fake_news.evaluation import evaluate_predictions, load_predictions, save_predictions

preds = pd.DataFrame({"doc_id": val.doc_id, "split": "val", "prob_fake": prob_fake})
path = save_predictions(preds, model="example", regime="random")
print(path)
evaluate_predictions(load_predictions("example", "random"), split="val")

/home/thanos/Fake_news_recognition/data/processed/predictions/example__random.csv


{'n': 9379,
 'accuracy': 0.5558161850943597,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'tn': 5213,
 'fp': 0,
 'fn': 4166,
 'tp': 0}

In [8]:
path.unlink()  # remove the example file again

## 5. More

- **Markers kept:** `load_clean(remove_markers=False)` gives the same articles with source markers such as
  "(Reuters)" left in, only for measuring how much they inflate accuracy.
- **What the cleaning did and why:** `notebooks/01-data-analysis.ipynb`, `notebooks/02-data-cleaning.ipynb`,
  `reports/cleaning/DECISIONS.md`.
- **Adding a split regime** (MinHash pair / Person 2 → Person 1, who commits `splits.csv`):

```python
from fake_news.data import load_clean
from fake_news.data.split import check_split, make_group_split, save_split_column

clean = load_clean(columns=[])
groups = ...  # pd.Series: doc_id -> near-duplicate group id (or K-means cluster id)
split = make_group_split(clean, groups)  # stratify=False for a few large clusters
check_split(split, labels=clean.set_index("doc_id").is_fake, groups=groups)  # raises if a group leaks
save_split_column("grouped", split)
```

- **Paths and constants:** `fake_news.config` (`SEED`, `SPLIT_FRACTIONS`, `SPLITS_CSV`, `PREDICTIONS_DIR`, ...).